![Kayak](https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png)

# 🗂️ Notebook 1 — Collecte de données & Pipeline Cloud

**Projet :** Plan Your Trip with Kayak — Jedha Bootcamp Data Engineering  
**Auteur :** Lahonde  
**Date :** Février 2026

---

## Objectif de ce notebook

Ce notebook couvre **toute la chaîne de collecte et de stockage des données** :

| Étape | Description | Technologie |
|-------|-------------|-------------|
| **1** | Coordonnées GPS des 35 villes | Nominatim API |
| **2** | Données météo sur 5 jours | OpenWeatherMap API |
| **3** | Score météo & classement des villes | Pandas |
| **4** | Scraping des hôtels | Booking.com · Selenium · BeautifulSoup |
| **5** | Stockage brut (Data Lake) | AWS S3 |
| **6** | ETL vers la base de données (Data Warehouse) | AWS RDS · PostgreSQL |

> 📌 **Le Notebook 2** (`02_Visualization.ipynb`) prend le relais pour les cartes Plotly.

---
## ⚙️ Étape 0 — Installation & Configuration

### Librairies utilisées

| Librairie | Rôle |
|-----------|------|
| `requests` | Appels HTTP vers les APIs |
| `pandas` | Manipulation des DataFrames |
| `selenium` | Pilotage de Chrome headless |
| `beautifulsoup4` | Parsing HTML |
| `boto3` | SDK AWS (S3, RDS) |
| `sqlalchemy` + `psycopg2` | Connexion PostgreSQL |
| `python-dotenv` | Chargement des clés depuis `.env` |
| `tqdm` | Barres de progression |

### Sécurité des clés API

Toutes les clés sensibles sont stockées dans un fichier **`.env`** (jamais commité sur GitHub) et chargées via `python-dotenv`.

In [ ]:
# Exécuter une seule fois pour installer les dépendances
!pip install requests pandas selenium beautifulsoup4 boto3 sqlalchemy \
             psycopg2-binary python-dotenv tqdm webdriver-manager -q

In [ ]:
# ── Librairies standard ──────────────────────────────────────────────
import os
import re
import io
import time
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

# ── Web Scraping ─────────────────────────────────────────────────────
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── AWS & Base de données ────────────────────────────────────────────
import boto3
from sqlalchemy import create_engine, text

# ── Chargement des variables d'environnement depuis .env ─────────────
load_dotenv()

OWM_API_KEY           = os.getenv("OWM_API_KEY")
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION            = os.getenv("AWS_DEFAULT_REGION", "eu-west-3")
S3_BUCKET             = os.getenv("S3_BUCKET")
RDS_URI               = os.getenv("RDS_URI")

# ── 35 villes françaises cibles ──────────────────────────────────────
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

print(f"Imports OK | {len(CITIES)} villes chargées")
print(f"OWM API Key : {'OK' if OWM_API_KEY else 'MANQUANTE'}")
print(f"AWS S3      : {'OK' if AWS_ACCESS_KEY_ID else 'MANQUANTE'}")
print(f"RDS URI     : {'OK' if RDS_URI and 'ENDPOINT' not in RDS_URI else 'MANQUANTE'}")

---
## 📍 Étape 1 — Coordonnées GPS avec Nominatim

### Pourquoi Nominatim ?

[Nominatim](https://nominatim.openstreetmap.org/) est le géocodeur officiel d'**OpenStreetMap**. Il permet de convertir un nom de lieu en coordonnées GPS **(latitude, longitude)**.

**Avantages :**
- ✅ Gratuit, sans abonnement
- ✅ Pas de clé API requise
- ✅ Très précis pour les villes françaises

**Contrainte :** respecter une **pause d'1 seconde** entre les requêtes (politique d'utilisation de Nominatim).

### Pourquoi a-t-on besoin des coordonnées GPS ?

L'API OpenWeatherMap demande des coordonnées GPS (lat/lon) et non des noms de villes. C'est pourquoi cette étape est obligatoire **avant** de collecter la météo.

In [ ]:
def get_coordinates(city_name: str) -> dict:
    """
    Récupère la latitude et longitude d'une ville via l'API Nominatim.
    
    Paramètres
    ----------
    city_name : str
        Nom de la ville française
    
    Retour
    ------
    dict : {'city': str, 'lat': float, 'lon': float} ou None si non trouvé
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": f"{city_name}, France",  # On précise France pour éviter les ambiguïtés
        "format": "json",
        "limit": 1                    # On ne veut que le 1er résultat
    }
    # Nominatim exige un User-Agent identifiant l'application
    headers = {"User-Agent": "KayakProjectJedha/1.0"}
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        data = response.json()
        if data:
            return {
                "city": city_name,
                "lat" : float(data[0]["lat"]),
                "lon" : float(data[0]["lon"])
            }
    except Exception as e:
        print(f"Erreur GPS pour {city_name} : {e}")
    return None


# ── Collecte pour les 35 villes ──────────────────────────────────────
coords_list = []
for city in tqdm(CITIES, desc="Géolocalisation"):
    result = get_coordinates(city)
    if result:
        coords_list.append(result)
    time.sleep(1)  # Respecter la politique Nominatim

df_coords = pd.DataFrame(coords_list)
df_coords["city_id"] = range(1, len(df_coords) + 1)

print(f"\n✅ {len(df_coords)} villes géolocalisées sur {len(CITIES)}")
df_coords.head(10)

---
## ☀️ Étape 2 — Données Météo avec OpenWeatherMap

### Choix de l'API

On utilise l'**API Forecast** (`/data/2.5/forecast`) d'OpenWeatherMap — **gratuite** et disponible sans abonnement supplémentaire.

> ⚠️ L'API "One Call 3.0" mentionnée dans le sujet nécessite désormais un abonnement payant. L'API Forecast couvre 5 jours de prévisions, ce qui est suffisant pour notre scoring météo.

### Données collectées par ville

L'API retourne des prévisions par créneaux de **3 heures** (40 points sur 5 jours). On les **agrège par jour** pour obtenir :

| Variable | Description |
|----------|-------------|
| `temp_max` | Température maximale du jour (°C) |
| `temp_min` | Température minimale du jour (°C) |
| `pop` | Probabilité de précipitations (0 à 1) |
| `rain_mm` | Volume de pluie total (mm) |
| `humidity` | Humidité moyenne (%) |
| `wind_speed` | Vitesse du vent moyenne (m/s) |

In [ ]:
def get_weather_forecast(lat: float, lon: float, city_id: int, city_name: str) -> list:
    """
    Récupère les prévisions météo sur 5 jours pour une ville.
    Agrège les données 3h en données journalières.
    
    Paramètres
    ----------
    lat, lon  : float — Coordonnées GPS de la ville
    city_id   : int   — Identifiant unique de la ville
    city_name : str   — Nom de la ville
    
    Retour
    ------
    list : Liste de dictionnaires, un par jour
    """
    url    = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "lat"  : lat,
        "lon"  : lon,
        "appid": OWM_API_KEY,
        "units": "metric",  # Celsius
        "lang" : "fr",      # Descriptions en français
        "cnt"  : 40         # 40 créneaux = 5 jours complets
    }
    
    try:
        r    = requests.get(url, params=params, timeout=10)
        data = r.json()
        if r.status_code != 200:
            print(f"Erreur OWM {city_name} : {data.get('message', r.status_code)}")
            return []
        
        # ── Agrégation des créneaux 3h par date ──────────────────────
        daily = {}
        for item in data.get("list", []):
            date = pd.to_datetime(item["dt"], unit="s").strftime("%Y-%m-%d")
            if date not in daily:
                daily[date] = {"temps": [], "humidity": [], "pop": [],
                               "rain_mm": 0, "wind_speeds": []}
            daily[date]["temps"].append(item["main"]["temp"])
            daily[date]["humidity"].append(item["main"]["humidity"])
            daily[date]["pop"].append(item.get("pop", 0))
            daily[date]["rain_mm"] += item.get("rain", {}).get("3h", 0)
            daily[date]["wind_speeds"].append(item["wind"]["speed"])
        
        # ── Construction des enregistrements journaliers ──────────────
        records = []
        for date, vals in daily.items():
            records.append({
                "city_id"   : city_id,
                "city"      : city_name,
                "date"      : date,
                "temp_min"  : round(min(vals["temps"]), 1),
                "temp_max"  : round(max(vals["temps"]), 1),
                "humidity"  : round(sum(vals["humidity"]) / len(vals["humidity"]), 1),
                "pop"       : round(sum(vals["pop"]) / len(vals["pop"]), 2),
                "rain_mm"   : round(vals["rain_mm"], 1),
                "wind_speed": round(sum(vals["wind_speeds"]) / len(vals["wind_speeds"]), 1),
            })
        return records
    
    except Exception as e:
        print(f"Erreur météo {city_name} : {e}")
        return []


# ── Collecte météo pour les 35 villes ────────────────────────────────
all_weather = []
for _, row in tqdm(df_coords.iterrows(), total=len(df_coords), desc="Météo"):
    records = get_weather_forecast(row["lat"], row["lon"], row["city_id"], row["city"])
    all_weather.extend(records)
    time.sleep(0.3)  # Éviter le rate-limit OWM

df_weather = pd.DataFrame(all_weather)

print(f"\n✅ {len(df_weather)} enregistrements météo collectés")
print(f"   Période : {df_weather['date'].min()} → {df_weather['date'].max()}")
print(f"   Villes  : {df_weather['city'].nunique()}")
df_weather.head(10)

---
## 🏆 Étape 3 — Score Météo & Classement

### Méthodologie

On crée un **score composite** pour classer les villes selon la qualité de leur météo sur 5 jours. 

**Pourquoi 4 critères ?**

| Critère | Poids | Justification |
|---------|-------|---------------|
| Température max moyenne | **40 %** | Critère principal du confort vacances |
| Probabilité de pluie | **30 %** | Personne ne veut de la pluie en vacances |
| Volume de pluie total | **20 %** | Un peu de pluie ≠ beaucoup de pluie |
| Humidité moyenne | **10 %** | Complète le critère température |

### Normalisation min-max

Pour comparer des variables d'unités différentes (°C, %, mm), on **normalise** chaque variable entre 0 et 100 :

$$\text{norm}(x) = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \times 100$$

Pour les critères où **plus petit = meilleur** (pluie, humidité), on inverse : `100 - norm(x)`.

In [ ]:
# ── Agrégation sur toute la période par ville ─────────────────────────
df_agg = df_weather.groupby(["city_id", "city"]).agg(
    temp_max_mean = ("temp_max",  "mean"),
    pop_mean      = ("pop",       "mean"),
    rain_total    = ("rain_mm",   "sum"),
    humidity_mean = ("humidity",  "mean"),
    wind_mean     = ("wind_speed","mean")
).reset_index()


def normalize(series: pd.Series, reverse: bool = False) -> pd.Series:
    """Normalisation min-max entre 0 et 100.
    Si reverse=True, les valeurs les plus faibles donnent le meilleur score."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)
    norm = (series - mn) / (mx - mn) * 100
    return (100 - norm) if reverse else norm


# ── Calcul des scores partiels ────────────────────────────────────────
df_agg["score_temp"] = normalize(df_agg["temp_max_mean"])
df_agg["score_rain"] = normalize(df_agg["pop_mean"],     reverse=True)
df_agg["score_vol"]  = normalize(df_agg["rain_total"],   reverse=True)
df_agg["score_hum"]  = normalize(df_agg["humidity_mean"],reverse=True)

# ── Score composite pondéré ───────────────────────────────────────────
df_agg["weather_score"] = (
    df_agg["score_temp"] * 0.40 +
    df_agg["score_rain"] * 0.30 +
    df_agg["score_vol"]  * 0.20 +
    df_agg["score_hum"]  * 0.10
).round(2)

# ── Classement final ──────────────────────────────────────────────────
df_ranking = df_agg.sort_values("weather_score", ascending=False).reset_index(drop=True)
df_ranking["rank"] = df_ranking.index + 1

# Fusionner avec les coordonnées GPS
df_weather_final = df_ranking.merge(df_coords[["city_id", "lat", "lon"]], on="city_id")

# Sauvegarde CSV
df_weather_final.to_csv("weather_cities.csv", index=False)

print("✅ Classement météo calculé — sauvegardé dans weather_cities.csv")
print()
print("🏆 TOP 10 DESTINATIONS :")
df_weather_final[
    ["rank", "city", "temp_max_mean", "pop_mean", "rain_total", "humidity_mean", "weather_score"]
].head(10)

---
## 🏨 Étape 4 — Scraping des Hôtels sur Booking.com

### Pourquoi Selenium et pas `requests` ?

Booking.com est un site **dynamique** : son contenu est généré par **JavaScript** après le chargement initial de la page. Un simple `requests.get()` renvoie une page quasi-vide, sans les fiches hôtels.

**Selenium** résout ce problème en pilotant un **vrai navigateur Chrome** (en mode headless = sans interface graphique). Chrome exécute le JavaScript, et on récupère ensuite le HTML complet avec **BeautifulSoup**.

### Processus de scraping

```
Pour chaque ville :
  1. Ouvrir Booking.com avec l'URL de recherche de la ville
  2. Attendre le chargement JS (4 secondes)
  3. Fermer le popup de cookies si présent
  4. Parser le HTML avec BeautifulSoup
  5. Extraire les 20 premières fiches hôtels
  6. Extraire : nom, URL, score, description
  7. Pause 2 secondes avant la ville suivante
```

### Données collectées

| Champ | Source HTML |
|-------|-------------|
| `hotel_name` | `a[data-testid='title-link']` |
| `url` | `href` du lien titre |
| `score` | `div[data-testid='review-score']` → regex `(\d+)[,.]\d+` |
| `description` | `span[data-testid='address']` |
| `lat` / `lon` | Coordonnées du centre-ville (GPS hôtel non exposé dans les cards) |

> 📌 **Note légale :** Le scraping est réalisé à des fins académiques uniquement.

In [ ]:
def create_driver() -> webdriver.Chrome:
    """Crée et retourne un driver Chrome headless (sans fenêtre)."""
    opts = Options()
    opts.add_argument("--headless")               # Pas d'interface graphique
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    # Simuler un vrai navigateur pour éviter la détection anti-bot
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )

In [ ]:
def scrape_city_hotels(driver, city_name: str, city_id: int,
                       city_lat: float, city_lon: float,
                       n_hotels: int = 20) -> list:
    """
    Scrape les hôtels d'une ville sur Booking.com.
    
    Paramètres
    ----------
    driver    : webdriver.Chrome — driver Selenium initialisé
    city_name : str   — Nom de la ville
    city_id   : int   — Identifiant unique
    city_lat  : float — Latitude du centre-ville
    city_lon  : float — Longitude du centre-ville
    n_hotels  : int   — Nombre d'hôtels à collecter (défaut : 20)
    
    Retour
    ------
    list : Liste de dicts avec les infos hôtels
    """
    hotels = []
    city_enc = city_name.replace(" ", "+")
    url = f"https://www.booking.com/searchresults.html?ss={city_enc}&lang=fr&dest_type=city"

    driver.get(url)
    time.sleep(4)  # Attendre le rendu JavaScript

    # Fermer le popup cookies si présent
    try:
        btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        btn.click()
        time.sleep(1)
    except Exception:
        pass

    # Attendre l'apparition des cards hôtels
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="property-card"]'))
        )
    except Exception:
        pass

    # Parser le HTML avec BeautifulSoup
    soup  = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.find_all("div", {"data-testid": "property-card"})

    for card in cards[:n_hotels]:
        # Nom de l'hôtel
        link_tag = card.find("a", {"data-testid": "title-link"})
        name = link_tag.get_text(strip=True) if link_tag else None
        if not name:
            continue

        # URL de la fiche hôtel
        href = link_tag.get("href", "") if link_tag else ""
        if href and not href.startswith("http"):
            href = "https://www.booking.com" + href
        hotel_url = href.split("?")[0] if ".html" in href else href

        # Score utilisateur — extrait depuis le texte "Avec une note de 9,2"
        score = None
        score_tag = card.find("div", {"data-testid": "review-score"})
        if score_tag:
            m = re.search(r"(\d+)[,.](\d+)", score_tag.get_text())
            if m:
                score = float(f"{m.group(1)}.{m.group(2)}")

        # Description / adresse
        addr_tag = card.find(["span", "div"], {"data-testid": "address"})
        description = addr_tag.get_text(strip=True) if addr_tag else ""

        hotels.append({
            "city_id"    : city_id,
            "city"       : city_name,
            "hotel_name" : name,
            "url"        : hotel_url,
            "lat"        : city_lat,   # Coords du centre-ville
            "lon"        : city_lon,
            "score"      : score,
            "description": description,
        })

    return hotels

In [ ]:
# ── Scraping sur les 35 villes (~9-10 minutes) ────────────────────────
driver     = create_driver()
all_hotels = []
np.random.seed(42)  # Pour la reproductibilité de la dispersion GPS

for _, row in tqdm(df_coords.iterrows(), total=len(df_coords), desc="Hotels Booking"):
    hotels = scrape_city_hotels(
        driver, row["city"], row["city_id"],
        row["lat"], row["lon"], n_hotels=20
    )
    all_hotels.extend(hotels)
    time.sleep(2)  # Pause pour éviter le blocage IP

driver.quit()

# ── Nettoyage ─────────────────────────────────────────────────────────
df_hotels = pd.DataFrame(all_hotels)
df_hotels["score"] = pd.to_numeric(df_hotels["score"], errors="coerce")

# Légère dispersion GPS autour du centre (±2 km) pour la carte
n = len(df_hotels)
df_hotels["lat"] += np.random.uniform(-0.02, 0.02, n)
df_hotels["lon"] += np.random.uniform(-0.02, 0.02, n)
df_hotels["hotel_id"] = range(1, n + 1)

# Sauvegarde
df_hotels.to_csv("hotels_booking.csv", index=False)

print(f"✅ {len(df_hotels)} hôtels collectés")
print(f"   Villes  : {df_hotels['city'].nunique()}")
print(f"   Avec score : {df_hotels['score'].notna().sum()}")
print()
print("Top 10 hôtels (par score) :")
(
    df_hotels.dropna(subset=["score"])
    .sort_values("score", ascending=False)
    [["hotel_name", "city", "score"]]
    .head(10)
)

---
## ☁️ Étape 5 — Data Lake sur AWS S3

### Concept : Data Lake vs Data Warehouse

| | Data Lake (S3) | Data Warehouse (RDS) |
|--|----------------|---------------------|
| **Données** | Brutes, non transformées | Nettoyées, structurées |
| **Format** | CSV, JSON, Parquet... | Tables SQL |
| **Usage** | Stockage, exploration, ML | Analyse, BI, requêtes |
| **Coût** | Très faible | Plus élevé |

### Architecture S3 du projet

```
s3://alh-consulting-kayak/
└── raw/
    ├── weather_cities.csv          (5.4 KB  — 35 villes, score météo)
    ├── hotels_booking.csv          (119 KB  — 679 hôtels)
    └── weather_daily_detail.csv    (12 KB   — détail 5j × 35 villes)
```

Le dossier `raw/` signifie que les données sont **brutes** — c'est la convention dans un Data Lake.

In [ ]:
# ── Connexion au client S3 via Boto3 ──────────────────────────────────
s3_client = boto3.client(
    "s3",
    aws_access_key_id     = AWS_ACCESS_KEY_ID,
    aws_secret_access_key = AWS_SECRET_ACCESS_KEY,
    region_name           = AWS_REGION
)

# ── Upload des fichiers CSV ───────────────────────────────────────────
files_to_upload = [
    "weather_cities.csv",
    "hotels_booking.csv",
    "weather_daily_detail.csv"
]

for filename in files_to_upload:
    if os.path.exists(filename):
        s3_client.upload_file(
            Filename = filename,
            Bucket   = S3_BUCKET,
            Key      = f"raw/{filename}"    # Stocké dans le dossier raw/
        )
        print(f"✅ s3://{S3_BUCKET}/raw/{filename}")

# ── Vérification du contenu du bucket ─────────────────────────────────
print(f"\nContenu du bucket '{S3_BUCKET}' :")
response = s3_client.list_objects_v2(Bucket=S3_BUCKET)
for obj in response.get("Contents", []):
    size_kb = obj["Size"] / 1024
    print(f"  {obj['Key']:<45} {size_kb:>8.1f} KB")

---
## 🗄️ Étape 6 — ETL vers AWS RDS (Data Warehouse)

### Qu'est-ce que l'ETL ?

**ETL = Extract · Transform · Load** — C'est le processus qui transforme les données brutes du Data Lake en données propres et structurées dans le Data Warehouse.

```
EXTRACT  ──→  Lire les CSV depuis S3
TRANSFORM ──→  Sélectionner, nettoyer, typer les colonnes
LOAD     ──→  Insérer dans PostgreSQL (AWS RDS)
```

### Instance RDS utilisée

| Paramètre | Valeur |
|-----------|--------|
| Identifiant | `kayak-jedha` |
| Endpoint | `kayak-jedha.cdwsi2uiosiy.eu-west-3.rds.amazonaws.com` |
| Moteur | PostgreSQL 16.6 |
| Classe | `db.t3.micro` |
| Région | `eu-west-3` (Paris) |
| Base | `kayak_db` |

### Schéma des tables

**Table `cities_weather`** (35 lignes)
```sql
city_id | city | lat | lon | temp_max_mean | pop_mean | rain_total | humidity_mean | weather_score | rank
```

**Table `hotels`** (679 lignes)
```sql
hotel_id | city_id | city | hotel_name | url | lat | lon | score | description
```

In [ ]:
# ── EXTRACT : Lire les données depuis S3 ─────────────────────────────
def read_csv_from_s3(s3_key: str) -> pd.DataFrame:
    """Lit un fichier CSV depuis S3 et retourne un DataFrame."""
    obj = s3_client.get_object(Bucket=S3_BUCKET, Key=s3_key)
    df  = pd.read_csv(io.BytesIO(obj["Body"].read()))
    print(f"  Extrait : {s3_key} → {df.shape[0]} lignes, {df.shape[1]} colonnes")
    return df

print("EXTRACT depuis S3 :")
df_weather_s3 = read_csv_from_s3("raw/weather_cities.csv")
df_hotels_s3  = read_csv_from_s3("raw/hotels_booking.csv")

In [ ]:
# ── TRANSFORM : Nettoyage et sélection des colonnes ───────────────────

# Table météo — on garde les colonnes utiles et on arrondit
df_weather_clean = df_weather_s3[[
    "city_id", "city", "lat", "lon",
    "temp_max_mean", "pop_mean", "rain_total",
    "humidity_mean", "weather_score", "rank"
]].copy()
df_weather_clean["temp_max_mean"] = df_weather_clean["temp_max_mean"].round(1)
df_weather_clean["weather_score"] = df_weather_clean["weather_score"].round(2)

# Table hôtels — on nettoie les scores
df_hotels_clean = df_hotels_s3[[
    "city_id", "city", "hotel_name", "url",
    "lat", "lon", "score", "description"
]].copy()
df_hotels_clean["score"] = pd.to_numeric(df_hotels_clean["score"], errors="coerce")
df_hotels_clean["hotel_id"] = range(1, len(df_hotels_clean) + 1)

print("TRANSFORM :")
print(f"  Table météo  : {df_weather_clean.shape}")
print(f"  Table hôtels : {df_hotels_clean.shape}")

# Aperçu de la table météo triée
df_weather_clean.sort_values("rank").head(10)

In [ ]:
# ── LOAD : Insertion dans PostgreSQL via SQLAlchemy ───────────────────
engine = create_engine(RDS_URI)

# Supprimer les tables si elles existent (pour rejouer le script proprement)
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS hotels"))
    conn.execute(text("DROP TABLE IF EXISTS cities_weather"))
    conn.commit()

# Insertion
df_weather_clean.to_sql("cities_weather", engine, if_exists="replace", index=False)
print(f"✅ Table 'cities_weather' créée : {len(df_weather_clean)} lignes")

df_hotels_clean.to_sql("hotels", engine, if_exists="replace", index=False)
print(f"✅ Table 'hotels' créée : {len(df_hotels_clean)} lignes")

In [ ]:
# ── Vérification : requêtes SQL directement sur RDS ───────────────────
print("Vérification depuis RDS PostgreSQL :")

with engine.connect() as conn:
    # Top 5 destinations
    top5 = pd.read_sql(
        text("SELECT rank, city, temp_max_mean, pop_mean, rain_total, weather_score "
             "FROM cities_weather ORDER BY rank LIMIT 5"),
        conn
    )
    print("\n🏆 Top 5 destinations (depuis RDS) :")
    display(top5)

    # Top 5 hôtels
    top_hotels = pd.read_sql(
        text("SELECT hotel_name, city, score FROM hotels "
             "WHERE score IS NOT NULL ORDER BY score DESC LIMIT 5"),
        conn
    )
    print("\n🏨 Top 5 hôtels (depuis RDS) :")
    display(top_hotels)

---
## ✅ Résumé — Ce que nous avons accompli

| Étape | Résultat | Fichiers |
|-------|---------|----------|
| GPS | 35 villes géolocalisées | `cities_coords.csv` |
| Météo | 175 prévisions (5j × 35 villes) | `weather_daily_detail.csv` |
| Score | Classement des 35 villes | `weather_cities.csv` |
| Scraping | 679 hôtels collectés | `hotels_booking.csv` |
| Data Lake | 3 CSV stockés sur S3 | `s3://alh-consulting-kayak/raw/` |
| Data Warehouse | 2 tables PostgreSQL | `cities_weather` + `hotels` |

### Infrastructure AWS

- **S3 Bucket** : `alh-consulting-kayak` — région `eu-west-3` (Paris)
- **RDS Endpoint** : `kayak-jedha.cdwsi2uiosiy.eu-west-3.rds.amazonaws.com`

---
> 👉 **Continuer avec** `02_Visualization.ipynb` pour les cartes Plotly interactives.